In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
sys.path.append(str(repo_root.parent / "src"))


In [2]:
from ingestion.load_redial import load_redial, quick_stats
import pandas as pd

In [3]:
df_redial = load_redial("../data/raw/ReDial.txt")
quick_stats(df_redial)
df_redial.head()


Rows: 22458
Conversations: 1000
Speakers: {'USER': 11806, 'SYSTEM': 10652}


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall
0,0,1,USER,Hello!,None,"4,3,4,3","[4, 3, 4, 3]",False
1,0,2,SYSTEM,hello!,None,None,None,False
2,0,3,USER,I'm looking for some drama movie suggestions.....,None,"3,3,4,3","[3, 3, 4, 3]",False
3,0,4,SYSTEM,"Have you ever seen the movie ""Veronica (2017)""",None,None,None,False
4,0,5,USER,"I haven't, I'll give it a quick search though!",None,"3,3,4,3","[3, 3, 4, 3]",False


In [4]:
# Calculate satisfaction_mean and add label column
def avg_score(score_list):
    if not score_list:
        return None
    return sum(score_list) / len(score_list)

df_redial["satisfaction_mean"] = df_redial["scores"].apply(avg_score)

# Add label column if dialog_act exists
if "dialog_act" in df_redial.columns:
    df_redial["label"] = df_redial["dialog_act"]


In [5]:
# REDIAL Statistics
import numpy as np

print("=" * 50)
print("REDIAL Dataset Statistics")
print("=" * 50)
print(f"Num rows: {len(df_redial)}")
print(f"Num conversations: {df_redial['conv_id'].nunique()}")
print(f"Num OVERALL lines: {df_redial['is_overall'].sum()}")
print(f"\nSpeakers value counts:")
print(df_redial['speaker'].value_counts())
if 'label' in df_redial.columns:
    print(f"\nLabel value counts:")
    print(df_redial['label'].value_counts())
else:
    print("\nLabel column does not exist")
satisfaction_mean = df_redial['satisfaction_mean'].mean()
if pd.notna(satisfaction_mean):
    print(f"\nSatisfaction mean: {satisfaction_mean:.4f}")
else:
    print("\nSatisfaction mean: N/A")
print("\n" + "=" * 50)
print("Preview head():")
print("=" * 50)
df_redial.head()


REDIAL Dataset Statistics
Num rows: 22458
Num conversations: 1000
Num OVERALL lines: 0

Speakers value counts:
speaker
USER      11806
SYSTEM    10652
Name: count, dtype: int64

Label value counts:
Series([], Name: count, dtype: int64)

Satisfaction mean: 3.1301

Preview head():


,conv_id,turn_id,speaker,text,dialog_act,scores_raw,scores,is_overall,satisfaction_mean,label
0,0,1,USER,Hello!,None,"4,3,4,3","[4, 3, 4, 3]",False,3.50,None
1,0,2,SYSTEM,hello!,None,None,None,False,NaN,None
2,0,3,USER,I'm looking for some drama movie suggestions.....,None,"3,3,4,3","[3, 3, 4, 3]",False,3.25,None
3,0,4,SYSTEM,"Have you ever seen the movie ""Veronica (2017)""",None,None,None,False,NaN,None
4,0,5,USER,"I haven't, I'll give it a quick search though!",None,"3,3,4,3","[3, 3, 4, 3]",False,3.25,None


In [6]:
df_redial.to_parquet("../data/processed/redial_turns.parquet", index=False)
print("Saved redial_turns.parquet")


Saved redial_turns.parquet
